# 05. Phase 2 Results

Two-model by three-architecture comparison on complete 300-query runs.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from notebooks.notebook_utils import *

set_plot_style()
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

phase2, phase2_summary = load_phase2_results()
phase2_effects = phase2_effect_sizes()
phase2_summary[['ci_low', 'ci_high']] = pd.DataFrame(phase2_summary['faithfulness_ci'].tolist(), index=phase2_summary.index)
markdown_df(phase2_summary[['model_family', 'architecture', 'queries', 'mean_faithfulness', 'ci_low', 'ci_high', 'mean_bertscore_f1', 'mean_hallucination_rate']])

## Factorial summary

In [ ]:
phase2_pivot = phase2_summary.pivot(index='architecture', columns='model_family', values='mean_faithfulness').round(3)
phase2_pivot

In [ ]:
sns.pointplot(data=phase2_summary, x='architecture', y='mean_faithfulness', hue='model_family', markers='o')
plt.title('Interaction plot: model family x architecture')
plt.tight_layout()

## Two-way ANOVA on per-question faithfulness

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols
anova_model = ols('faithfulness ~ C(model_family) * C(architecture)', data=phase2).fit()
sm.stats.anova_lm(anova_model, typ=2).round(4)

## Effect sizes

In [ ]:
markdown_df(phase2_effects[['model_family', 'comparison', 'mean_difference', 'cohens_d', 'wilcoxon_p']])

In [ ]:
phase2['architecture_order'] = pd.Categorical(phase2['architecture'], categories=['Simple RAG', 'Advanced RAG', 'Long Context'], ordered=True)
sns.boxplot(data=phase2, x='architecture_order', y='faithfulness', hue='model_family')
plt.xlabel('Architecture')
plt.title('Per-question faithfulness distributions')
plt.tight_layout()